# DeepSeek-Coder-V2-Lite-Instruct (abliterated) — Colab chat v5: model picker • context & reasoning picker • web search (Parallel MCP) • public link

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (preselected if you opened this file in Colab).
2. **Cell 1** — pick the model size from the dropdown (default **Q4_K_M**, balanced), then run it. It **mounts Google Drive first** (`USE_DRIVE_CACHE` is on by default), then builds llama.cpp and downloads the model and tools *in parallel* (~10–15 min the first time; re-running is fast).
3. **Cell 2** — pick the **context window** (default **163,840**, the model's maximum), **reasoning** (leave it on **Off** — this model answers directly) and **web search** (default **Parallel**, free, no key), then run it. It shows the chat right in the notebook, plus a private link and a **public link**.
4. **Keep this tab open and leave Cell 2 running.** It watches the model server and the link and repairs them automatically.

**Good to know**
- **Model sizes** in this repo: Q8_0, Q6_K, Q5_K_M, Q5_K_S, Q4_K_M, Q4_K_S, IQ4_XS, Q3_K_L, Q3_K_M, Q3_K_S, Q2_K (by mradermacher). On a free T4, Q8_0 does not fit and Q6_K only fits with a smaller context; Cell 2 then uses the largest size that fits and tells you. **Q4_K_M fits the full 163,840.**
- **Reasoning:** this model answers directly (it has no `<think>` block), so **Off** is the right choice; the picker is kept from v5 — the other levels only matter for models that think. You can also switch it per chat in the UI.
- **Web search:** the model gets `web_search` and `web_fetch` tools. Parallel is free without a key; if it's busy, DuckDuckGo/Exa are tried automatically. The first search asks permission in the chat — click **Always allow** — or use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button to skip all prompts. It is on by default, remembered in your browser, and applies instantly without reloading — an answer that is already streaming keeps streaming.
- **Public link:** uses a **Cloudflare** link (no account, new address each run). The chat inside the notebook always works too.
- **Long input:** 1,000+ lines of text or code is fine — paste it or attach the file.
- **About the model:** 16B-parameter MoE coder (2.4B active) from DeepSeek, trained for 338 programming languages, abliterated by criscar22 and quantized by mradermacher. "Abliterated" = the refusal behaviour was removed (uncensored); use responsibly and only where that is allowed.
- Chats are stored in your browser tab. If the public link changes, it's a new origin, so export chats you care about from the UI settings.


In [ ]:
#@title 1️⃣ Setup — choose the model size, then run (≈10–15 min the first time) { display-mode: "form" }
MODEL_QUANT = "Q5_K_S · 11.1 GB · high quality, a bit smaller" #@param ["Q4_K_M · 10.4 GB · balanced, most room for context (default)", "Q8_0 · 16.7 GB · best quality (needs a bigger GPU than a free T4)", "Q6_K · 14.1 GB · near-lossless (needs a bigger GPU than a free T4)", "Q5_K_M · 11.9 GB · high quality", "Q5_K_S · 11.1 GB · high quality, a bit smaller", "Q4_K_S · 9.5 GB · balanced, a bit smaller", "IQ4_XS · 8.6 GB · balanced, I-quant", "Q3_K_L · 8.5 GB · small", "Q3_K_M · 8.1 GB · small", "Q3_K_S · 7.5 GB · small", "Q2_K · 6.4 GB · smallest, lowest quality"]
USE_DRIVE_CACHE = True #@param {type:"boolean"}
# USE_DRIVE_CACHE: mounts Google Drive first and saves the compiled server there so the next session skips the ~10 min build.

import os, sys, time, json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

# ---------- Google Drive first (optional cache; must mount on the main thread) ----------
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("✅ Google Drive mounted (build cache)")
    except Exception as e:
        USE_DRIVE_CACHE = False
        print(f"⚠️ Could not mount Google Drive ({e}); continuing without the build cache.")


QUANT     = MODEL_QUANT.split()[0]                     # e.g. "Q8_0"
REPO_ID   = "mradermacher/DeepSeek-Coder-V2-Lite-Instruct-abliterated-GGUF"
FILENAME  = f"DeepSeek-Coder-V2-Lite-Instruct-abliterated.{QUANT}.gguf"
LLAMA_TAG = "b11925"                                   # pinned llama.cpp release (tested), so a new upstream change can't break this notebook

LLAMA_DIR = "/content/llama.cpp"
BIN_PATH  = f"{LLAMA_DIR}/build/bin/llama-server"
STAMP     = f"{LLAMA_DIR}/build/colab_chat_build_stamp.txt"
MODEL_DIR = "/content/models"
CF_PATH   = "/content/cloudflared"
CONFIG    = "/content/dscoder_v2lite_config.json"
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(MODEL_DIR, exist_ok=True)

# ---------- GPU check ----------
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                   "--format=csv,noheader"], text=True).strip().splitlines()[0]
    gpu_name, cap, vram = [x.strip() for x in gpu.split(",")]
    ARCH = cap.replace(".", "")
    print(f"✅ GPU: {gpu_name} (sm_{ARCH}, {vram})")
except Exception:
    raise RuntimeError("❌ No GPU found. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

free_gb = shutil.disk_usage("/content").free / 1e9
if free_gb < 25:
    print(f"⚠️ Only {free_gb:.0f} GB of disk free; the build + model need ~20 GB.")

# ---------- optional Drive cache (must mount on the main thread) ----------
CACHE_DIR  = "/content/drive/MyDrive/colab_chat_cache"
CACHED_BIN = f"{CACHE_DIR}/llama-server-{LLAMA_TAG}-sm{ARCH}"
if USE_DRIVE_CACHE:
    os.makedirs(CACHE_DIR, exist_ok=True)

def run(cmd, log, env=None, retries=1):
    for attempt in range(retries):
        with open(log, "a") as f:
            f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        if p.returncode == 0:
            return
        if attempt + 1 < retries:
            time.sleep(5)
    tail = open(log, errors="replace").read()[-4000:]
    raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{tail}")

def binary_ok(path):
    try:
        out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60)
        return out.returncode == 0
    except Exception:
        return False

def build_server():
    want = f"{LLAMA_TAG} sm{ARCH}"
    if os.path.exists(BIN_PATH) and os.path.exists(STAMP) and open(STAMP).read().strip() == want and binary_ok(BIN_PATH):
        return "already built"
    if USE_DRIVE_CACHE and os.path.exists(CACHED_BIN):
        os.makedirs(os.path.dirname(BIN_PATH), exist_ok=True)
        shutil.copy(CACHED_BIN, BIN_PATH); os.chmod(BIN_PATH, 0o755)
        if binary_ok(BIN_PATH):
            open(STAMP, "w").write(want)
            return "restored from Drive cache"
    log = "/content/build.log"
    open(log, "w").close()
    shutil.rmtree(LLAMA_DIR, ignore_errors=True)        # always start from the pinned, clean source
    run(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG,
         "https://github.com/ggml-org/llama.cpp", LLAMA_DIR], log, retries=3)
    # Web UI: use the official prebuilt UI of this exact release (no Node.js build needed)
    dist = f"{LLAMA_DIR}/tools/ui/dist"
    os.makedirs(dist, exist_ok=True)
    try:
        run(["wget", "-q", "--tries=3", "-O", "/content/llama-ui.tar.gz",
             f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/llama-{LLAMA_TAG}-ui.tar.gz"], log)
        run(["tar", "-xzf", "/content/llama-ui.tar.gz", "-C", dist, "--strip-components=1"], log)
    except Exception:
        shutil.rmtree(dist, ignore_errors=True)         # cmake then fetches the same UI version from Hugging Face
    env = dict(os.environ, HF_UI_VERSION=LLAMA_TAG)
    run(["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build",
         "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={ARCH}",
         "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_OPENSSL=OFF",
         "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
         "-DCMAKE_BUILD_TYPE=Release"], log, env=env)
    run(["cmake", "--build", f"{LLAMA_DIR}/build", "--config", "Release",
         "--target", "llama-server", "-j", str(os.cpu_count() or 2)], log, env=env)
    if not binary_ok(BIN_PATH):
        raise RuntimeError("The server was built but does not start. See /content/build.log")
    open(STAMP, "w").write(want)
    if USE_DRIVE_CACHE:
        shutil.copy(BIN_PATH, CACHED_BIN)
    return "built from source"

def get_cloudflared():
    if os.path.exists(CF_PATH) and os.path.getsize(CF_PATH) > 1_000_000 and binary_ok(CF_PATH):
        return "already there"
    run(["wget", "-q", "--tries=3", "-O", CF_PATH,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        "/content/cf.log", retries=3)
    os.chmod(CF_PATH, 0o755)
    if os.path.getsize(CF_PATH) < 1_000_000 or not binary_ok(CF_PATH):
        os.remove(CF_PATH)
        raise RuntimeError("cloudflared download looks broken; run this cell again.")
    return "downloaded"

def get_web_deps():
    # DuckDuckGo search + page-to-text, used by the DuckDuckGo option and as automatic fallback
    p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ddgs", "trafilatura"],
                       capture_output=True, text=True)
    return "installed" if p.returncode == 0 else "skipped (pip failed; Parallel/Exa still work)"

def download_model():
    try:
        import huggingface_hub  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download, list_repo_files
    token = None
    try:  # optional Colab secret HF_TOKEN (only needed if the repo ever becomes gated)
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    fname = FILENAME
    try:
        files = list_repo_files(REPO_ID, token=token)
        if fname not in files:
            cands = sorted(f for f in files if f.lower().endswith(".gguf")
                           and "mmproj" not in f.lower() and QUANT.lower() in f.lower())
            if not cands:
                raise RuntimeError(f"No {QUANT} file in {REPO_ID}. Files:\n" + "\n".join(files))
            fname = cands[0]
    except RuntimeError:
        raise
    except Exception as e:
        print(f"   (could not list repo files: {e}; trying the standard file name)")
    # free disk: remove other quants from earlier runs (keeps the image projector)
    for old in os.listdir(MODEL_DIR):
        if old.endswith(".gguf") and old != os.path.basename(fname) and "mmproj" not in old:
            os.remove(os.path.join(MODEL_DIR, old))
    def fetch(name):
        last = None
        for attempt in range(4):
            try:
                return hf_hub_download(REPO_ID, name, local_dir=MODEL_DIR, token=token)
            except Exception as e:
                last = e
                time.sleep(10 * (attempt + 1))
        raise RuntimeError(f"Download of {name} failed after 4 tries: {last}")
    model = fetch(fname)
    mmproj = ""
    return model, mmproj

t0 = time.time()
print(f"⏳ Building llama.cpp {LLAMA_TAG} and downloading {FILENAME} at the same time...")
with ThreadPoolExecutor(max_workers=4) as ex:
    jobs = {"build": ex.submit(build_server), "download": ex.submit(download_model),
            "tunnel tool": ex.submit(get_cloudflared), "web tools": ex.submit(get_web_deps)}
    last = -1
    while not all(j.done() for j in jobs.values()):
        time.sleep(5)
        mins = int((time.time() - t0) // 60)
        if mins != last:
            last = mins
            print(f"   {mins:>2} min | " + " | ".join(f"{k}: {'done' if j.done() else '…'}" for k, j in jobs.items()))
    errors = []
    for k, j in jobs.items():
        if j.exception() is not None and k != "web tools":
            errors.append(f"[{k}] {j.exception()}")
    if errors:
        raise RuntimeError("Setup failed — just run this cell again (finished parts are reused).\n\n" + "\n\n".join(errors))
    build_status = jobs["build"].result()
    model_path, mmproj_path = jobs["download"].result()
    cf_status = jobs["tunnel tool"].result()
    web_status = jobs["web tools"].result() if jobs["web tools"].exception() is None else "skipped"

json.dump({"model": model_path, "mmproj": mmproj_path, "quant": QUANT, "bin": BIN_PATH,
           "cloudflared": CF_PATH, "tag": LLAMA_TAG, "ui_tag": LLAMA_TAG}, open(CONFIG, "w"))
print(f"\n✅ Server:      {build_status} (llama.cpp {LLAMA_TAG})")
print(f"✅ Model:       {os.path.basename(model_path)} ({os.path.getsize(model_path)/1e9:.1f} GB)")
if mmproj_path:
    print(f"✅ Vision:      {os.path.basename(mmproj_path)}")
print(f"✅ Tunnel tool: {cf_status}")
print(f"✅ Web tools:   {web_status}")
print(f"⏱  {int(time.time() - t0)//60} min {int(time.time() - t0)%60} s.  👉 Now run Cell 2.")


In [ ]:
#@title 2️⃣ Launch — choose context, reasoning and web search, then run (keep this cell running!) { display-mode: "form" }
CONTEXT_WINDOW = "163840 · max (model limit, default)" #@param ["163840 · max (model limit, default)", "131072", "98304", "65536", "32768", "16384", "8192", "4096"]
REASONING = "Max · unlimited thinking" #@param ["Off · answers directly, no thinking (default)", "Max · unlimited thinking", "High · up to 8192 thinking tokens", "Medium · up to 2048 thinking tokens", "Low · up to 512 thinking tokens"]
WEB_SEARCH = "Parallel · free, no key needed (default)" #@param ["Parallel · free, no key needed (default)", "Exa · free tier is rate-limited, best with a key", "DuckDuckGo · free, no key needed", "Off · no web tools"]
AUTO_FALLBACK = True #@param {type:"boolean"}
PARALLEL_API_KEY = "VtYAbqEv6_sOvxUtcGeHI28jHmIpGuxM_talrLr6" #@param {type:"string"}
EXA_API_KEY = "29334ebe-08ca-4c98-8b57-af84986d1464" #@param {type:"string"}
PUBLIC_LINK = "Cloudflare · no account, new link each run (default)" #@param ["Cloudflare · no account, new link each run (default)", "Off · private Colab link only"]
SHOW_INLINE = True #@param {type:"boolean"}
# CONTEXT_WINDOW: if your choice doesn't fit in GPU memory, the next smaller size is used automatically (you'll be told).
# AUTO_FALLBACK: if the chosen search service is down or rate-limited, the other free ones are tried automatically.
# API keys are optional (you can also add them as Colab secrets named PARALLEL_API_KEY / EXA_API_KEY).
# A Parallel key raises limits and adds a deep "research" tool.
# PUBLIC_LINK: anyone with the link can use your session. Cloudflare needs no account; the address changes each run.
# SHOW_INLINE: also shows the chat right here in the notebook (private, always works).

# ---------------- advanced (rarely needed) ----------------
KV_CACHE_TYPE    = "q8_0"      # "q8_0" halves context memory vs "f16" with almost no quality loss
TEMP, TOP_P, TOP_K, MIN_P = 0.3, 0.95, 40, 0.0   # the model card's recommended sampling
PRESENCE_PENALTY = 0.0         # if replies ever loop/repeat, set 1.0
REPETITION_PENALTY = 1.0           # raise only if replies ever repeat
MAX_SERVER_RESTARTS = 5
# -----------------------------------------------------------

import os, re, sys, json, time, datetime, subprocess, shutil
import requests
from IPython.display import display, HTML

import socket
def free_port(start):
    """First port from `start` that nothing else (e.g. Colab's own services) is using."""
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sk:
            try:
                sk.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"No free port found near {start}")
LOG, TLOG = "/content/llama-server.log", "/content/tunnel.log"
CONFIG = "/content/dscoder_v2lite_config.json"
WEB_SCRIPT, MCP_CONFIG = "/content/dscoder_v2lite_web_mcp.py", "/content/dscoder_v2lite_mcp.json"
CTX_STEPS = [163840, 131072, 98304, 65536, 32768, 16384, 8192, 4096]
MIN_FREE_MIB = 400            # GPU memory that must stay free after loading, so long chats don't crash later
LOCAL = requests.Session()
LOCAL.trust_env = False       # talk to the local server/tunnel directly, never through a proxy

if not os.path.exists(CONFIG):
    raise RuntimeError("Run Cell 1 first.")
cfg = json.load(open(CONFIG))
BIN, MODEL_PATH, MMPROJ_PATH, CF, QUANT = cfg["bin"], cfg["model"], cfg.get("mmproj", ""), cfg["cloudflared"], cfg["quant"]
for p in (BIN, MODEL_PATH):
    if not os.path.exists(p):
        raise RuntimeError(f"Missing {p}. Run Cell 1 again.")
LINK_MODE = PUBLIC_LINK.split()[0].lower()            # cloudflare | off

# ---------------- parse the form ----------------
TARGET_CONTEXT = int(re.match(r"\d+", CONTEXT_WINDOW.strip()).group())
_r = REASONING.split()[0].lower()
THINKING = _r != "off"
REASONING_BUDGET = {"max": -1, "high": 8192, "medium": 2048, "low": 512}.get(_r, -1)   # same levels as the chat UI
WEB_BACKEND = {"parallel": "parallel", "exa": "exa", "duckduckgo": "duckduckgo"}.get(WEB_SEARCH.split()[0].lower(), "off")

def secret(name, typed):
    if typed.strip():
        return typed.strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""
TOGGLE_JS = r'''/* Injected by the notebook (tpuchat-v2): "Allow tool calls" switch inside the chat box,
   right after the + button. Applies instantly - never reloads the page, so a running
   answer (text or reasoning) is never interrupted. The choice is saved per browser. */
(function () {
  "use strict";
  if (window.__tpuchatAllowTools) return;
  window.__tpuchatAllowTools = true;

  var STATE_KEY = "tpuchat.allowTools";
  var TOOL_NAMES = ["web_search", "web_fetch", "web_research"];
  /* localStorage lists the official UI reads exactly once, at startup */
  var STORE_KEYS = ["LlamaUi.alwaysAllowedTools", "LlamaCppWebui.alwaysAllowedTools"];
  /* permission keys the UI derives for these tools (toolKey(): server:/mcp-<id>:/custom:) */
  var PERM_KEYS = [];
  var JUNK_KEYS = [];
  TOOL_NAMES.forEach(function (n) {
    PERM_KEYS.push("server:" + n, "mcp-web:" + n, "mcp-web:" + n.replace(/^web_/, ""), "custom:" + n);
  });
  ["web_web_search", "web_web_fetch", "web_web_research"].forEach(function (n) {
    JUNK_KEYS.push("server:" + n, "mcp-web:" + n, "custom:" + n);
  });
  TOOL_NAMES.forEach(function (n) {
    JUNK_KEYS.push("browser:" + n, "mcp:" + n);
  });

  function isOn() {
    try { var v = localStorage.getItem(STATE_KEY); return v === null ? true : v === "1"; }
    catch (e) { return true; }
  }
  function readList(k) {
    try {
      var a = JSON.parse(localStorage.getItem(k) || "[]");
      return Array.isArray(a) ? a.filter(function (x) { return typeof x === "string"; }) : [];
    } catch (e) { return []; }
  }
  function writeList(k, list) {
    try { localStorage.setItem(k, JSON.stringify(list)); } catch (e) {}
  }
  function cleanAndWrite(k, set) {
    JUNK_KEYS.forEach(function (p) { delete set[p]; });
    writeList(k, Object.keys(set));
  }
  function syncPerms(on) {
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      PERM_KEYS.forEach(function (p) { if (on) set[p] = 1; else delete set[p]; });
      cleanAndWrite(k, set);
    });
  }
  function allowSaved(tool) {
    var keys = ["server:" + tool, "mcp-web:" + tool, "mcp-web:" + tool.replace(/^web_/, ""), "custom:" + tool];
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      keys.forEach(function (p) { set[p] = 1; });
      cleanAndWrite(k, set);
    });
  }
  /* runs before the UI boots: the app picks this list up when it initializes */
  syncPerms(isOn());

  /* -------------- live plumbing (no reload: streaming is never interrupted) -------------- */

  var sessionAll = false;            /* "allow this session" covers all web tools until OFF again */
  var seenCard = { tool: "", t: 0 }; /* built-in permission card the user may act on */

  function isWebTool(t) { return typeof t === "string" && /^web_/.test(t); }

  /* Fetch gate - active only while the switch is OFF. Covers the one case localStorage
     cannot: web tools the UI still holds in its in-memory allowed set from an earlier
     boot, which would otherwise run silently instead of asking. */
  var origFetch = window.fetch.bind(window);
  window.fetch = function () {
    try {
      var input = arguments[0], init = arguments[1];
      var url = typeof input === "string" ? input : (input && input.url) || "";
      var method = ((init && init.method) || (input && input.method) || "GET").toUpperCase();
      if (method === "POST" && /\/tools\/?$/.test(String(url).split("?")[0]) &&
          init && typeof init.body === "string") {
        var tool = null;
        try { tool = (JSON.parse(init.body) || {}).tool; } catch (e) {}
        if (isWebTool(tool) && !isOn() && !sessionAll &&
            !(seenCard.tool === tool && Date.now() - seenCard.t < 8000)) {
          return gate(tool, arguments);
        }
      }
    } catch (e) { /* never break the app */ }
    return origFetch.apply(null, arguments);
  };

  var activeFinish = null;   /* decision callback of the currently open prompt */
  var gateChain = Promise.resolve();
  function gate(tool, args) {
    var run = function () {
      return new Promise(function (resolve) {
        var settled = false;
        var signal = args[1] && args[1].signal;
        function finish(allow, remember) {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          if (remember) sessionAll = true;
          if (remember === "saved") allowSaved(tool);
          if (allow) resolve(origFetch.apply(null, args));
          else resolve(new Response(JSON.stringify({
                   error: "Blocked by the user: the Allow tool calls switch is OFF in the chat box. Turn it ON (green) to run web tools without asking."
                 }), { status: 403, statusText: "Blocked",
                      headers: { "Content-Type": "application/json" } }));
        }
        function abortNow() {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          resolve(Promise.reject(new DOMException("The tool call was aborted.", "AbortError")));
        }
        if (signal) {
          if (signal.aborted) { abortNow(); return; }
          signal.addEventListener("abort", abortNow, { once: true });
        }
        showGateCard(tool, finish);
      });
    };
    gateChain = gateChain.then(run, run);
    return gateChain;
  }

  function esc(s) {
    return String(s).replace(/[&<>"']/g, function (c) {
      return { "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c];
    });
  }

  var gateEl = null;
  function showGateCard(tool, finish) {
    hideGateCard();
    var d = document.createElement("div");
    d.id = "tpu-gate-card";
    d.setAttribute("role", "dialog");
    d.setAttribute("aria-label", "Tool call permission");
    d.innerHTML = '<div class="t">DeepSeek-Coder-V2-Lite wants to run <b>' + esc(tool) + "</b></div>" +
      '<div class="s">The "Allow tool calls" switch is OFF, so this call needs your OK.</div>' +
      '<div class="row">' +
      '<button type="button" data-a="once">Allow once</button>' +
      '<button type="button" data-a="session">Allow this session</button>' +
      '<button type="button" data-a="saved">Always allow</button>' +
      '<button type="button" data-a="deny" class="deny">Deny</button>' +
      "</div>";
    d.addEventListener("click", function (e) {
      var b = e.target && e.target.closest ? e.target.closest("button[data-a]") : null;
      if (!b) return;
      var a = b.getAttribute("data-a");
      finish(a !== "deny", a === "session" ? "session" : (a === "saved" ? "saved" : null));
    });
    (document.body || document.documentElement).appendChild(d);
    gateEl = d;
    activeFinish = finish;
  }
  function hideGateCard() {
    if (gateEl) { try { gateEl.remove(); } catch (e) {} gateEl = null; }
  }
  document.addEventListener("keydown", function (e) {
    if (e.key === "Escape" && activeFinish) activeFinish(false, null);
  }, true);

  /* Watch the built-in permission card ("Allow use of <tool> ..."). While the switch is
     ON, approve it automatically (only needed when the in-memory set is missing keys,
     e.g. the switch was turned ON mid-session). While OFF, just remember it so a call
     the user just approved by hand is never asked twice. */
  function scanForCard() {
    var btns = document.getElementsByTagName("button");
    for (var i = 0; i < btns.length; i++) {
      var b = btns[i];
      if ((b.textContent || "").trim() !== "Allow once") continue;
      var root = b.parentElement, cardText = "";
      for (var k = 0; k < 7 && root; k++, root = root.parentElement) {
        cardText = root.textContent || "";
        if (cardText.indexOf("Allow use of ") >= 0) break;
      }
      if (cardText.indexOf("Allow use of ") < 0) continue;
      var m = cardText.match(/Allow use of\s+([A-Za-z0-9_.-]+)/);
      if (m) { seenCard.tool = m[1]; seenCard.t = Date.now(); }
      if (isOn() && !b.dataset.tpuDone) {
        b.dataset.tpuDone = "1";
        try { b.click(); } catch (e) {}
      }
    }
  }

  /* -------------- the switch, rendered inside the composer after the + button -------------- */

  var CSS_TEXT =
    "#tpu-allow-tools{display:inline-flex;align-items:center;gap:7px;height:32px;padding:0 12px;" +
    "border-radius:999px;border:1px solid rgba(148,163,184,.35);background:rgba(100,116,139,.14);" +
    "color:inherit;opacity:.85;font:600 12px/1 ui-sans-serif,system-ui,-apple-system,sans-serif;" +
    "cursor:pointer;user-select:none;white-space:nowrap;flex:0 0 auto;" +
    "transition:border-color .15s,background .15s,opacity .15s}" +
    "#tpu-allow-tools:hover{opacity:1;border-color:rgba(148,163,184,.6)}" +
    "#tpu-allow-tools .dot{width:8px;height:8px;border-radius:50%;background:#64748b;flex:0 0 auto;" +
    "transition:background .15s,box-shadow .15s}" +
    "#tpu-allow-tools.on{border-color:rgba(34,197,94,.55);background:rgba(34,197,94,.13);opacity:1}" +
    "#tpu-allow-tools.on .dot{background:#22c55e;box-shadow:0 0 7px rgba(34,197,94,.8)}" +
    "#tpu-allow-tools.on .lbl{color:#16a34a}" +
    "#tpu-allow-tools .lbl-short{display:none}" +
    "#tpu-gate-card{position:fixed;left:50%;bottom:96px;transform:translateX(-50%);z-index:2147483000;" +
    "width:min(440px,calc(100vw - 24px));padding:14px 16px;border-radius:14px;color:#e2e8f0;" +
    "background:#0f172a;border:1px solid rgba(148,163,184,.3);box-shadow:0 12px 40px rgba(0,0,0,.45);" +
    "font:400 13px/1.5 ui-sans-serif,system-ui,-apple-system,sans-serif;animation:tpu-pop .16s ease-out}" +
    "#tpu-gate-card .t{font-weight:600;margin-bottom:2px}" +
    "#tpu-gate-card .s{opacity:.75;margin-bottom:10px}" +
    "#tpu-gate-card .row{display:flex;flex-wrap:wrap;gap:8px}" +
    "#tpu-gate-card button{all:unset;cursor:pointer;padding:6px 12px;border-radius:8px;" +
    "font:600 12px/1 ui-sans-serif,system-ui,sans-serif;color:inherit;" +
    "border:1px solid rgba(148,163,184,.4);background:rgba(148,163,184,.12)}" +
    "#tpu-gate-card button:hover{background:rgba(148,163,184,.24)}" +
    "#tpu-gate-card button.deny{border-color:rgba(239,68,68,.5);color:#f87171}" +
    "@keyframes tpu-pop{from{opacity:0;transform:translateX(-50%) translateY(6px)}" +
    "to{opacity:1;transform:translateX(-50%) translateY(0)}}" +
    "@media (prefers-color-scheme: dark){#tpu-allow-tools.on .lbl{color:#4ade80}}" +
    "@media (prefers-color-scheme: light){#tpu-gate-card{background:#fff;color:#0f172a;" +
    "border-color:#cbd5e1;box-shadow:0 12px 40px rgba(15,23,42,.18)}}" +
    "@media (max-width:640px){#tpu-allow-tools{height:28px;padding:0 9px;gap:5px;font-size:11px}" +
    "#tpu-allow-tools .lbl-long{display:none}#tpu-allow-tools .lbl-short{display:inline}" +
    "#tpu-gate-card{bottom:84px}}";

  var chip = null;
  function paint() {
    if (!chip) return;
    var on = isOn();
    chip.classList.toggle("on", on);
    chip.setAttribute("aria-checked", on ? "true" : "false");
    var long = chip.querySelector(".lbl-long"), short = chip.querySelector(".lbl-short");
    if (long) long.textContent = "Allow tool calls: " + (on ? "ON" : "OFF");
    if (short) short.textContent = "Tools: " + (on ? "ON" : "OFF");
  }
  function buildChip() {
    try {
      var css = document.createElement("style");
      css.id = "tpu-allow-tools-css";
      css.textContent = CSS_TEXT;
      document.head.appendChild(css);
    } catch (e) {}
    var b = document.createElement("button");
    b.id = "tpu-allow-tools";
    b.type = "button";
    b.setAttribute("role", "switch");
    b.setAttribute("aria-label", "Allow tool calls");
    b.title = "ON: web tools (search / fetch) run without asking. OFF: each web tool call asks first. " +
      "Applies instantly - no reload, typing and generating continue undisturbed.";
    b.innerHTML = '<span class="dot"></span><span class="lbl lbl-long"></span><span class="lbl lbl-short"></span>';
    /* keep the composer textarea focused and never trigger the + menu or a form submit */
    ["mousedown", "pointerdown", "touchstart", "touchend"].forEach(function (ev) {
      b.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); });
    });
    b.addEventListener("click", function (e) {
      e.preventDefault();
      e.stopPropagation();
      var next = !isOn();
      try { localStorage.setItem(STATE_KEY, next ? "1" : "0"); } catch (err) {}
      syncPerms(next);
      if (!next) sessionAll = false;   /* a fresh OFF period asks again */
      paint();
      /* turning ON while a call is waiting for permission: let it through */
      if (next && activeFinish) activeFinish(true, "session");
      try {
        window.dispatchEvent(new StorageEvent("storage", { key: STATE_KEY, newValue: next ? "1" : "0" }));
      } catch (err) {}
    });
    paint();
    return b;
  }
  function ensureChip() {
    var host = document.querySelector("button.file-upload-button");
    if (!host || !host.isConnected) return false;
    if (!chip) chip = buildChip();
    /* preferred spot: the left group that holds the + button, next to it */
    var group = host.closest(".mr-auto") || host.parentElement;
    if (!group) return false;
    if (chip.isConnected && chip.parentElement === group) return true;
    try { group.appendChild(chip); paint(); } catch (e) { return false; }
    return true;
  }

  window.addEventListener("storage", function (e) {
    if (e && e.key === STATE_KEY) { try { paint(); } catch (err) {} }
  });

  /* -------------- keep-alive: survive Svelte re-renders and route changes -------------- */

  var scanTimer = 0, chipTimer = 0;
  function scheduleScan() {
    if (scanTimer) return;
    scanTimer = setTimeout(function () { scanTimer = 0; try { scanForCard(); } catch (e) {} }, 120);
  }
  function scheduleChip() {
    if (chipTimer) return;
    chipTimer = setTimeout(function () { chipTimer = 0; try { ensureChip(); } catch (e) {} }, 150);
  }
  function start() {
    try {
      var mo = new MutationObserver(function (muts) {
        scheduleChip();
        for (var i = 0; i < muts.length; i++) {
          var nodes = muts[i].addedNodes;
          for (var j = 0; j < nodes.length; j++) {
            var n = nodes[j];
            if (n.nodeType === 1 && (n.textContent || "").indexOf("Allow use of ") >= 0) {
              scheduleScan();
              return;
            }
          }
        }
      });
      mo.observe(document.documentElement, { childList: true, subtree: true });
    } catch (e) {}
    setInterval(function () {
      try { ensureChip(); } catch (e) {}
      try { scanForCard(); } catch (e) {}
    }, 1500);
    try { ensureChip(); } catch (e) {}
    console.info("[tpuchat] Allow tool calls switch installed (in the chat box, next to +)");
  }
  if (document.readyState === "loading") document.addEventListener("DOMContentLoaded", start);
  else start();
})();'''


WEBUI_PATH = ""

UI_MARK = "tpuchat-v2"

def _patch_webui(index_path):
    try:
        with open(index_path, encoding="utf-8") as f:
            html = f.read()
    except OSError:
        return False
    if UI_MARK in html:
        return True
    # strip any toggle script injected by older notebook versions, then add the current one
    while True:
        stripped = re.sub(r"<script>/\* Injected by the notebook.*?</script>\s*", "", html, count=1, flags=re.S)
        if stripped == html:
            break
        html = stripped
    if "</head>" not in html:
        return False
    try:
        with open(index_path, "w", encoding="utf-8") as f:
            f.write(html.replace("</head>", "<script>" + TOGGLE_JS + "</script>\n</head>", 1))
        return True
    except OSError:
        return False

def ensure_webui():
    """Copy/patch the official web UI so it has the Allow tool calls toggle."""
    global WEBUI_PATH
    target = "/content/webui"
    index = os.path.join(target, "index.html")
    if os.path.isfile(index):
        try:
            if UI_MARK in open(index, encoding="utf-8").read():
                WEBUI_PATH = target
                return WEBUI_PATH
        except OSError:
            pass
    dist = "/content/llama.cpp/tools/ui/dist"
    try:
        if not os.path.isfile(os.path.join(dist, "index.html")):
            tag = cfg.get("ui_tag") or "b11925"
            print(f"\u23f3 Downloading the chat UI ({tag})...")
            r = requests.get(
                f"https://github.com/ggml-org/llama.cpp/releases/download/{tag}/llama-{tag}-ui.tar.gz",
                timeout=180)
            r.raise_for_status()
            tgz = "/content/webui.tar.gz"
            with open(tgz, "wb") as f:
                f.write(r.content)
            shutil.rmtree(target, ignore_errors=True)
            os.makedirs(target, exist_ok=True)
            subprocess.run(["tar", "-xzf", tgz, "-C", target, "--strip-components=1"], check=True)
            if _patch_webui(os.path.join(target, "index.html")):
                print("\u2705 Chat UI ready (with the Allow tool calls toggle)")
                WEBUI_PATH = target
                return WEBUI_PATH
        else:
            shutil.rmtree(target, ignore_errors=True)
            shutil.copytree(dist, target)
            if _patch_webui(os.path.join(target, "index.html")):
                print("\u2705 Chat UI ready (with the Allow tool calls toggle)")
                WEBUI_PATH = target
                return WEBUI_PATH
    except Exception as e:
        print(f"\u26a0\ufe0f Could not prepare the enhanced UI ({e}); using the built-in UI.")
    WEBUI_PATH = ""
    return WEBUI_PATH

if LINK_MODE not in ("cloudflare", "off"):
    LINK_MODE = "cloudflare"
if LINK_MODE == "cloudflare" and not os.path.exists(CF):
    raise RuntimeError("cloudflared is missing. Run Cell 1 again (or set PUBLIC_LINK to Off).")
PKEY = secret("PARALLEL_API_KEY", PARALLEL_API_KEY) if WEB_BACKEND != "off" else ""
EKEY = secret("EXA_API_KEY", EXA_API_KEY) if WEB_BACKEND != "off" else ""

# ---------------- web tools (stdio MCP server run by llama-server) ----------------
WEB_MCP_SOURCE = r'''#!/usr/bin/env python3
"""Web tools: a tiny stdio MCP server for llama-server (--mcp-servers-config).

Gives the model two tools, web_search and web_fetch, backed by:
  parallel   - Parallel MCP (https://mcp.parallel.ai/mcp), free without a key
  exa        - Exa MCP (https://mcp.exa.ai/mcp), free tier is rate limited, key optional
  duckduckgo - DuckDuckGo search (ddgs) + direct page download, no key
If the chosen backend fails (rate limit, network), it falls back to the next one
automatically, so a search almost never comes back empty.

Settings come from environment variables set by the notebook:
  WEB_BACKEND        parallel | exa | duckduckgo        (default parallel)
  PARALLEL_API_KEY   optional, higher limits + adds web_research
  EXA_API_KEY        optional
  WEB_FALLBACK       1 = try other backends on failure  (default 1)
  WEB_MAX_CHARS      cap on characters returned per call (default 24000)
Only JSON-RPC goes to stdout; diagnostics go to stderr.
"""
import json, os, re, sys, time, html, urllib.request, urllib.error

BACKEND = os.environ.get("WEB_BACKEND", "parallel").strip().lower() or "parallel"
PARALLEL_KEY = os.environ.get("PARALLEL_API_KEY", "").strip()
EXA_KEY = os.environ.get("EXA_API_KEY", "").strip()
FALLBACK = os.environ.get("WEB_FALLBACK", "1") != "0"
MAX_CHARS = int(os.environ.get("WEB_MAX_CHARS", "24000") or 24000)
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"


def log(*a):
    print("[colab-web]", *a, file=sys.stderr, flush=True)


# ----------------------------------------------------------------- remote MCP client
class RemoteMCP:
    """Minimal Streamable-HTTP MCP client (JSON or SSE replies)."""

    def __init__(self, url, headers=None, timeout=90):
        self.url, self.timeout, self.sid = url, timeout, None
        self.headers = {"Content-Type": "application/json",
                        "Accept": "application/json, text/event-stream",
                        "User-Agent": "colab-chat/4"}
        self.headers.update(headers or {})
        self._id = 0

    def _post(self, payload, want_reply=True):
        h = dict(self.headers)
        if self.sid:
            h["Mcp-Session-Id"] = self.sid
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode("utf-8"),
                                     headers=h, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                sid = r.headers.get("Mcp-Session-Id")
                if sid:
                    self.sid = sid
                ctype = (r.headers.get("Content-Type") or "").lower()
                body = r.read().decode("utf-8", errors="replace")   # always UTF-8 (fixes mojibake)
        except urllib.error.HTTPError as e:
            detail = ""
            try:
                detail = e.read().decode("utf-8", errors="replace")[:500]
            except Exception:
                pass
            raise RuntimeError(f"HTTP {e.code} from {self.url}: {detail}")
        if not want_reply:
            return None
        msgs = []
        if "event-stream" in ctype:
            data_lines = []
            for line in body.splitlines() + [""]:          # SSE: blank line ends an event
                if line.startswith("data:"):
                    data_lines.append(line[5:].lstrip(" "))
                elif line.strip() == "":
                    if data_lines:
                        try:
                            msgs.append(json.loads("\n".join(data_lines)))
                        except ValueError:
                            pass
                        data_lines = []
        elif body.strip():
            j = json.loads(body)
            msgs = j if isinstance(j, list) else [j]
        for m in msgs:
            if isinstance(m, dict) and m.get("id") == payload.get("id"):
                return m
        for m in msgs:                                      # some servers reply with id null on errors
            if isinstance(m, dict) and "error" in m:
                return m
        raise RuntimeError("No reply from MCP server")

    def _rpc(self, method, params=None):
        self._id += 1
        reply = self._post({"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}})
        if "error" in reply:
            err = reply["error"]
            raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
        return reply.get("result", {})

    def call(self, tool, args):
        self._rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                 "clientInfo": {"name": "colab-chat", "version": "4"}})
        try:
            self._post({"jsonrpc": "2.0", "method": "notifications/initialized"}, want_reply=False)
        except Exception:
            pass
        res = self._rpc("tools/call", {"name": tool, "arguments": args})
        text = result_text(res)
        if res.get("isError"):
            raise RuntimeError(text or "tool error")
        return text


def result_text(res):
    parts = []
    for c in res.get("content") or []:
        if isinstance(c, dict) and c.get("type") == "text" and c.get("text"):
            parts.append(c["text"])
    if not parts and res.get("structuredContent") is not None:
        parts.append(json.dumps(res["structuredContent"], ensure_ascii=False, indent=1))
    return "\n".join(parts).strip()


# ----------------------------------------------------------------- backends
def _queries(a):
    q = a.get("query") or ""
    qs = a.get("search_queries") or []
    if isinstance(qs, str):
        qs = [qs]
    qs = [str(x).strip() for x in qs if str(x).strip()]
    if q.strip() and q.strip() not in qs:
        qs.insert(0, q.strip())
    return qs[:5]


def _urls(a):
    u = a.get("urls") or a.get("url") or []
    if isinstance(u, str):
        u = re.split(r"[\s,]+", u)
    out = []
    for x in u:
        x = str(x).strip().strip("<>\"'")
        if x and not re.match(r"^https?://", x, re.I):
            x = "https://" + x
        if x and x not in out:
            out.append(x)
    return out[:10]


def parallel_search(a):
    qs = _queries(a)
    objective = (a.get("objective") or "").strip() or (qs[0] if qs else "")
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr).call(
        "web_search", {"objective": objective[:4000], "search_queries": qs or [objective[:200]]})


def parallel_fetch(a):
    args = {"urls": _urls(a)}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:200]
    if a.get("full_content"):
        args["full_content"] = True
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr).call("web_fetch", args)


def parallel_research(a):
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"}
    args = {"input": str(a.get("question") or a.get("input") or a.get("query") or "")[:20000]}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr, timeout=280).call("web_research", args)


def _exa():
    hdr = {"Authorization": f"Bearer {EXA_KEY}"} if EXA_KEY else {}
    return RemoteMCP("https://mcp.exa.ai/mcp", hdr)


def exa_search(a):
    qs = _queries(a)
    args = {"query": " ".join(qs[:1]) or a.get("objective", ""), "numResults": 8}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:4000]
    return _exa().call("web_search_exa", args)


def exa_fetch(a):
    return _exa().call("web_fetch_exa", {"urls": _urls(a),
                                         "maxCharacters": 60000 if a.get("full_content") else 8000})


def ddg_search(a):
    try:
        from ddgs import DDGS
    except ImportError:
        raise RuntimeError("ddgs is not installed (pip install ddgs)")
    out, seen = [], set()
    for q in _queries(a)[:3] or [a.get("objective", "")]:
        for r in DDGS().text(q, max_results=8) or []:
            url = r.get("href") or r.get("url")
            if not url or url in seen:
                continue
            seen.add(url)
            out.append(f"## {r.get('title','')}\nURL: {url}\n{r.get('body','')}")
    if not out:
        raise RuntimeError("DuckDuckGo returned no results")
    return "\n\n".join(out)


def _download(url):
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html,*/*;q=0.8"})
    with urllib.request.urlopen(req, timeout=30) as r:
        raw = r.read(6_000_000)
        cs = r.headers.get_content_charset() or "utf-8"
        ctype = (r.headers.get("Content-Type") or "").lower()
    txt = raw.decode(cs, errors="replace")
    if "html" not in ctype and "<html" not in txt[:2000].lower():
        return txt
    try:
        import trafilatura
        ext = trafilatura.extract(txt, url=url, include_links=False, include_tables=True,
                                  output_format="markdown")
        if ext and len(ext) > 200:
            return ext
    except Exception:
        pass
    txt = re.sub(r"(?is)<(script|style|noscript|svg|head)[^>]*>.*?</\1>", " ", txt)
    txt = re.sub(r"(?s)<[^>]+>", " ", txt)
    return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n\s*\n+", "\n\n", html.unescape(txt))).strip()


def direct_fetch(a):
    urls, parts = _urls(a), []
    if not urls:
        raise RuntimeError("no valid URL given")
    per = max(3000, MAX_CHARS // len(urls))
    for u in urls:
        try:
            parts.append(f"# {u}\n{_download(u)[:per]}")
        except Exception as e:
            parts.append(f"# {u}\n(could not load: {e})")
    return "\n\n".join(parts)


SEARCH = {"parallel": parallel_search, "exa": exa_search, "duckduckgo": ddg_search}
FETCH = {"parallel": parallel_fetch, "exa": exa_fetch, "duckduckgo": direct_fetch}


def run_with_fallback(table, args):
    order = [BACKEND] + ([b for b in ("parallel", "duckduckgo", "exa") if b != BACKEND] if FALLBACK else [])
    errors = []
    for b in order:
        if b not in table:
            continue
        for attempt in range(2):                     # one quick retry per backend
            try:
                text = table[b](args)
                if text and text.strip():
                    note = "" if b == BACKEND else f"[results from {b}; {BACKEND} was unavailable]\n"
                    return note + text
                raise RuntimeError("empty result")
            except Exception as e:
                msg = str(e)
                log(f"{b} attempt {attempt + 1} failed: {msg[:300]}")
                if "rate limit" in msg.lower() or "HTTP 4" in msg:
                    break                            # don't hammer a rate-limited backend
                time.sleep(1.5)
        errors.append(f"{b}: {msg[:200]}")
    raise RuntimeError("all web backends failed - " + " | ".join(errors))


# ----------------------------------------------------------------- MCP server side
TOOLS = [
    {"name": "search",
     "description": ("Search the web for current information. Returns titles, URLs and text excerpts. "
                     "Use for anything recent, factual, or that you are unsure about."),
     "inputSchema": {"type": "object", "properties": {
         "query": {"type": "string", "description": "Short keyword search query, 3-8 words."},
         "objective": {"type": "string", "description": "Optional: one sentence on what you want to find."}},
         "required": ["query"]}},
    {"name": "fetch",
     "description": "Read one or more web pages (e.g. URLs from web_search) and return their main text.",
     "inputSchema": {"type": "object", "properties": {
         "urls": {"type": "array", "items": {"type": "string"}, "description": "Page URLs to read (1-5)."},
         "objective": {"type": "string", "description": "Optional: what to look for on the pages."},
         "full_content": {"type": "boolean", "description": "Return the whole page instead of the relevant parts."}},
         "required": ["urls"]}},
]
if BACKEND == "parallel" and PARALLEL_KEY:
    TOOLS.append({"name": "research",
                  "description": "Deep research on one question; returns a cited answer (slower, uses your Parallel credits).",
                  "inputSchema": {"type": "object", "properties": {
                      "question": {"type": "string", "description": "The research question."}},
                      "required": ["question"]}})


def handle(msg):
    method, mid = msg.get("method"), msg.get("id")
    if mid is None:
        return None                                          # notification
    if method == "initialize":
        pv = (msg.get("params") or {}).get("protocolVersion") or "2025-06-18"
        return {"protocolVersion": pv, "capabilities": {"tools": {"listChanged": False}},
                "serverInfo": {"name": "colab-web", "version": "4.0"}}
    if method == "ping":
        return {}
    if method == "tools/list":
        return {"tools": TOOLS}
    if method == "tools/call":
        p = msg.get("params") or {}
        name, args = p.get("name"), p.get("arguments") or {}
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                args = {"query": args}
        try:
            if name in ("search", "web_search"):
                text = run_with_fallback(SEARCH, args)
            elif name in ("fetch", "web_fetch"):
                if not _urls(args):
                    raise RuntimeError("web_fetch needs at least one URL in 'urls'")
                text = run_with_fallback(FETCH, args)
            elif name in ("research", "web_research"):
                text = parallel_research(args)
            else:
                raise RuntimeError(f"unknown tool {name}")
            if len(text) > MAX_CHARS:
                text = text[:MAX_CHARS] + "\n\n[...truncated]"
            return {"content": [{"type": "text", "text": text}], "isError": False}
        except Exception as e:
            return {"content": [{"type": "text", "text": f"Web tool error: {e}"}], "isError": True}
    raise KeyError(method)


def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except ValueError:
            continue
        msgs = msg if isinstance(msg, list) else [msg]
        for m in msgs:
            if not isinstance(m, dict):
                continue
            try:
                res = handle(m)
                if res is None:
                    continue
                out = {"jsonrpc": "2.0", "id": m.get("id"), "result": res}
            except KeyError:
                out = {"jsonrpc": "2.0", "id": m.get("id"),
                       "error": {"code": -32601, "message": f"Method not found: {m.get('method')}"}}
            except Exception as e:
                out = {"jsonrpc": "2.0", "id": m.get("id"), "error": {"code": -32603, "message": str(e)}}
            sys.stdout.write(json.dumps(out, ensure_ascii=False) + "\n")
            sys.stdout.flush()


if __name__ == "__main__":
    main()
'''

web_ok = False
if WEB_BACKEND != "off":
    open(WEB_SCRIPT, "w").write(WEB_MCP_SOURCE)
    web_env = {"WEB_BACKEND": WEB_BACKEND, "WEB_FALLBACK": "1" if AUTO_FALLBACK else "0",
               "PARALLEL_API_KEY": PKEY, "EXA_API_KEY": EKEY, "PYTHONIOENCODING": "utf-8"}
    json.dump({"mcpServers": {"web": {"command": sys.executable, "args": ["-u", WEB_SCRIPT],
                                      "env": web_env, "timeout_ms": 300000}}}, open(MCP_CONFIG, "w"))
    print(f"⏳ Testing web search ({WEB_BACKEND})... ", end="", flush=True)
    try:
        probe = "\n".join(json.dumps(m) for m in [
            {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "search", "arguments": {"query": "llama.cpp github"}}}]) + "\n"
        out = subprocess.run([sys.executable, "-u", WEB_SCRIPT], input=probe, capture_output=True, text=True,
                             timeout=120, env=dict(os.environ, **web_env)).stdout
        res = [json.loads(l) for l in out.splitlines() if l.strip().startswith("{")]
        r2 = next((r for r in res if r.get("id") == 2), {}).get("result", {})
        if r2 and not r2.get("isError"):
            web_ok = True
            txt = r2["content"][0]["text"]
            print("✅" + (f"  (note: {txt.splitlines()[0][1:-1]})" if txt.startswith("[results from") else ""))
        else:
            msg = (r2.get("content") or [{"text": "no reply"}])[0]["text"]
            print(f"⚠️ {msg[:300]}\n   The tools stay enabled and retry on every call.")
            web_ok = True
    except Exception as e:
        print(f"⚠️ test failed ({e}); the tools stay enabled and retry on every call.")
        web_ok = True

# ---------------- model server ----------------
today = datetime.date.today().strftime("%A, %B %d, %Y")
system_msg = f"You are DeepSeek-Coder-V2-Lite-Instruct (abliterated), a helpful coding assistant. Today's date is {today}."
if WEB_BACKEND != "off":
    system_msg += (" You have web tools: use web_search for anything recent, factual or uncertain, then web_fetch "
                   "to read the most relevant pages, and cite the URLs you used.")
UI_CONFIG = json.dumps({"systemMessage": system_msg})

def pkill(name):
    subprocess.run(["pkill", "-x", name], stderr=subprocess.DEVNULL)

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()

def start_server(ctx, kv):
    cmd = [BIN, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(PORT),
           "--alias", f"DeepSeek-Coder-V2-Lite-{QUANT}",
           "-ngl", "99", "--fit", "off", "-c", str(ctx), "--parallel", "1",
           "-fa", "on", "-ctk", kv, "-ctv", kv, "--cache-ram", "2048",
           "--jinja", "--ui-config", UI_CONFIG,
           "--temp", str(TEMP), "--top-p", str(TOP_P), "--top-k", str(TOP_K), "--min-p", str(MIN_P)]
    if THINKING:
        cmd += ["--reasoning", "on", "--reasoning-budget", str(REASONING_BUDGET)]
        if REASONING_BUDGET > 0:
            cmd += ["--reasoning-budget-message", " Okay, I have thought about this enough. Time to give the final answer.\n"]
    else:
        cmd += ["--reasoning", "off"]
    if PRESENCE_PENALTY > 0:
        cmd += ["--presence-penalty", str(PRESENCE_PENALTY)]
    if REPETITION_PENALTY != 1.0:
        cmd += ["--repeat-penalty", str(REPETITION_PENALTY)]
    if MMPROJ_PATH and os.path.exists(MMPROJ_PATH):
        cmd += ["--mmproj", MMPROJ_PATH]
    if WEB_BACKEND != "off":
        cmd += ["--mcp-servers-config", MCP_CONFIG]
    if WEBUI_PATH and os.path.isdir(WEBUI_PATH):
        cmd += ["--path", WEBUI_PATH]
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait_ready(proc, timeout=600):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            return False
        try:
            if LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                time.sleep(1)
                return proc.poll() is None
        except requests.RequestException:
            pass
        time.sleep(2)
    return False

def log_text():
    try:
        return open(LOG, errors="replace").read()
    except OSError:
        return ""

def fail_reason():
    log = log_text().lower()
    if any(s in log for s in ("out of memory", "cudamalloc failed", "failed to allocate", "unable to allocate")):
        return "out of memory"
    if "flash" in log and ("quantiz" in log or "requires" in log or "not supported" in log):
        return "flash-attention / KV-quantization problem"
    if "couldn't bind" in log or "address already in use" in log:
        return "port busy"
    if "error while handling argument" in log or "invalid argument" in log:
        return "bad argument: " + log.strip().splitlines()[-1][:200]
    return "see log"

def fully_on_gpu():
    m = re.search(r"offloaded (\d+)/(\d+) layers to gpu", log_text().lower())
    return m is None or m.group(1) == m.group(2)

def gpu_mem():
    try:
        used, total = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                                      "--format=csv,noheader,nounits"], capture_output=True, text=True
                                     ).stdout.strip().splitlines()[0].split(",")
        return int(used), int(total)
    except Exception:
        return None, None

def mem_label(kv):
    # the cache that stores the conversation; this is NOT the model file
    return "8-bit chat memory" if kv == "q8_0" else "16-bit chat memory"

def attempts_from(ctx):
    sizes = [ctx] + [c for c in CTX_STEPS if c < ctx]
    att = [(c, KV_CACHE_TYPE) for c in sizes]
    if KV_CACHE_TYPE != "f16":
        att += [(c, "f16") for c in (16384, 8192, 4096) if c <= ctx]   # last-resort fallbacks
    return att

def launch_with_best_context(start_ctx):
    global PORT
    skip_quantized = False
    for ctx, kv in attempts_from(start_ctx):
        if skip_quantized and kv != "f16":
            continue
        print(f"   trying {ctx:>7,} tokens ({mem_label(kv)}) ... ", end="", flush=True)
        proc = start_server(ctx, kv)
        ok, reason = wait_ready(proc), ""
        if ok and not fully_on_gpu():
            ok, reason = False, "model did not fully fit on the GPU"
        elif ok:
            used, total = gpu_mem()
            if used is not None and total - used < MIN_FREE_MIB:
                ok, reason = False, f"fits, but only {total-used} MiB GPU memory left (unsafe for long chats)"
        elif not ok:
            reason = fail_reason()
        if ok:
            print("✅")
            return proc, ctx, kv
        print(f"❌ ({reason})")
        if reason.startswith("flash"):
            skip_quantized = True
        if reason == "port busy":           # not a memory problem: pick another port and retry this size
            stop(proc)
            PORT = free_port(PORT + 1)
            print(f"   (port was busy, switching to {PORT})")
            proc = start_server(ctx, kv)
            if wait_ready(proc) and fully_on_gpu():
                print(f"   trying {ctx:>7,} tokens again on port {PORT} ... ✅")
                return proc, ctx, kv
            reason = fail_reason()
            print(f"   ❌ ({reason})")
        if reason.startswith("bad argument") or reason == "port busy":
            break
        stop(proc)
        time.sleep(3)
    print(log_text()[-3000:])
    raise RuntimeError("Could not start the model — see the log above. Try a smaller model in Cell 1 (e.g. Q4_K_M).")

# ---------------- public link ----------------
def tunnel_ready():
    try:
        return LOCAL.get(f"http://127.0.0.1:{METRICS_PORT}/ready", timeout=3).status_code == 200
    except requests.RequestException:
        return False

def url_answers(url):
    try:
        r = requests.get(url + "/health", timeout=8)
        return r.status_code == 200
    except requests.RequestException:
        return False

def start_cloudflare():
    pkill("cloudflared")
    time.sleep(1)
    for attempt in range(3):
        p = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                              "--protocol", "http2", "--no-autoupdate",
                              "--metrics", f"127.0.0.1:{METRICS_PORT}"],
                             stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(60):
            if p.poll() is not None:
                break
            try:
                found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
                         if not u.startswith("https://api.")]
            except OSError:
                found = []
            if found:
                url = found[0]
                break
            time.sleep(1)
        if url:
            for _ in range(30):   # wait until the link actually answers
                if p.poll() is not None:
                    break
                if tunnel_ready() and url_answers(url):
                    return p, url
                time.sleep(2)
            if p.poll() is None and tunnel_ready():
                return p, url          # connected; the edge may just need a few more seconds
        stop(p)
        time.sleep(3)
    return None, None

def link_alive(handle):
    if handle is None:
        return False
    return handle.poll() is None and tunnel_ready()

def start_link():
    return start_cloudflare()

def stop_link(handle):
    if handle is not None:
        stop(handle)

def private_url():
    try:
        from google.colab.output import eval_js
        return eval_js(f"google.colab.kernel.proxyPort({PORT}, {{'cache': false}})")
    except Exception:
        return None

def show_link(url, title):
    display(HTML(f'<div style="font-size:16px;margin:8px 0">{title} <b><a href="{url}" target="_blank">{url}</a></b></div>'))

# ================= main =================
pkill("llama-server")
pkill("cloudflared")
time.sleep(2)
PORT, METRICS_PORT = free_port(8090), free_port(20241)
WEBUI_PATH = ensure_webui()

print(f"⏳ Loading DeepSeek-Coder-V2-Lite {QUANT} with up to {TARGET_CONTEXT:,} tokens of context...")
server, CTX, KV = launch_with_best_context(TARGET_CONTEXT)

used, total = gpu_mem()
think_txt = ("OFF" if not THINKING else "MAX (unlimited)" if REASONING_BUDGET < 0
             else f"{REASONING.split()[0]} (up to {REASONING_BUDGET:,} tokens)")
print(f"\n✅ Model ready  |  model: {QUANT}  |  context: {CTX:,} tokens ({mem_label(KV)})  |  reasoning: {think_txt}")
if CTX < TARGET_CONTEXT:
    print(f"   ℹ️ {TARGET_CONTEXT:,} didn't fit in GPU memory with {QUANT}; using {CTX:,}. "
          f"For more context pick a smaller model in Cell 1 (Q4_K_M fits the full 163,840).")
print(f"   web search: {('ON · ' + WEB_BACKEND + (' + auto-fallback' if AUTO_FALLBACK else '')) if WEB_BACKEND != 'off' else 'OFF'}"
      f"  |  vision: {'ON' if MMPROJ_PATH else 'off'}  |  GPU memory: {used} / {total} MiB\n")

tunnel, public_url = None, None
try:
    from google.colab import output
    # Colab's private new-tab links (*.prod.colab.dev) now return 404 in modern browsers,
    # so only the in-notebook view is shown; use the public link for a full tab.
    if SHOW_INLINE:
        print("💬 Chat inside the notebook (private, only you):")
        output.serve_kernel_port_as_iframe(PORT, height=850)
    if LINK_MODE != "off":
        print(f"⏳ Opening the public link ({LINK_MODE})...")
        tunnel, public_url = start_link()
        if public_url:
            show_link(public_url, "🌍 Public link:")
            print("   (If it says it can't be reached, wait 30 s and refresh, or open it in an incognito window.)")
        else:
            print("⚠️ The public link didn't start yet — it keeps retrying below. The chat above works meanwhile.")

    print("\nHow to use: open a link → type your message. Long text/code (1,000+ lines) is fine: paste it or attach the file.")
    if WEB_BACKEND != "off":
        print("Web search: use the “Allow tool calls: ON / OFF” switch inside the chat box, right after the + button to control whether searches ask for permission (applies instantly, no reload).")
    print("Reasoning can also be changed per chat in the UI (the 💡 / reasoning button); “Default” = your choice above.")
    print("Keep this cell running. Press ■ (stop) to shut everything down.\n")

    start, last_report = time.time(), 0
    server_restarts = tunnel_restarts = bad_ticks = 0
    while True:
        time.sleep(20)

        # 1) model server (auto-restart; if it ran out of memory, restart with the next smaller context)
        if server.poll() is not None:
            if server_restarts >= MAX_SERVER_RESTARTS:
                print("❌ Model server keeps crashing. Last log lines:\n")
                print(log_text()[-3000:])
                break
            server_restarts += 1
            if fail_reason() == "out of memory":
                smaller = [c for c in CTX_STEPS if c < CTX]
                if smaller:
                    print(f"⚠️ Ran out of GPU memory at {CTX:,} tokens — restarting with {smaller[0]:,}.")
                    CTX = smaller[0]
            print(f"⚠️ Model server stopped — restarting ({server_restarts}/{MAX_SERVER_RESTARTS})...")
            server = start_server(CTX, KV)
            if wait_ready(server):
                print("✅ Model server is back. Your chats are kept in your browser; just resend the last message.")
            continue

        # 2) public link
        if LINK_MODE != "off":
            bad_ticks = 0 if link_alive(tunnel) else bad_ticks + 1
            if tunnel is None or bad_ticks >= 3:
                tunnel_restarts += 1
                print(f"⚠️ Public link dropped — reconnecting (#{tunnel_restarts})...")
                stop_link(tunnel)
                tunnel, new_url = start_link()
                bad_ticks = 0
                if new_url and new_url != public_url:
                    public_url = new_url
                    show_link(public_url, "🌍 NEW public link (the old one no longer works):")
                elif new_url:
                    print("✅ Public link reconnected (same address).")

        # 3) heartbeat every 10 min
        mins = int((time.time() - start) // 60)
        if mins - last_report >= 10:
            last_report = mins
            print(f"⏱ Up {mins} min | server restarts: {server_restarts} | link restarts: {tunnel_restarts}")
except KeyboardInterrupt:
    print("Stopping...")
finally:
    stop_link(tunnel)
    stop(server)
    print("Shut down.")


## Troubleshooting
- **"No GPU found"** → Runtime → Change runtime type → T4 GPU → Save, run Cell 1 again.
- **Context is lower than you picked** → that's the most your GPU can hold with that model size. Pick **Q4_K_M** (or smaller) in Cell 1 and re-run Cells 1 and 2 for the full 163,840.
- **Web search says rate-limited** → it falls back automatically. For heavy use, add a free key from platform.parallel.ai in `PARALLEL_API_KEY` (or a Colab secret with that name).
- **Tool prompts keep appearing / never appear** → use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button. On (green) = web tools run without asking; Off = every web tool call asks first (allow once, for this session, or always). The choice is saved in your browser.
- **The Allow tool calls switch is missing next to the + button (or the old floating button still shows)** → hard-refresh the page (Ctrl+Shift+R / Cmd+Shift+R) once. The notebook serves the patched UI from `/content/webui`; a previously cached UI may need one refresh.
- **Replies loop or repeat** → set `PRESENCE_PENALTY = 1.0` or raise `REPETITION_PENALTY` in Cell 2.
- **Public link doesn't open** → wait ~30 s and refresh, or try an incognito window. The chat shown inside the notebook always works.
- **"q8_0 / 8-bit chat memory"** → that's the conversation cache, not the model. Your chosen model size is shown as `model:`.
- **Download or build failed** → just run Cell 1 again; finished parts are reused. Logs: `/content/build.log`.
- **Session disconnected** → reconnect, run Cell 1 then Cell 2 (`USE_DRIVE_CACHE` is on, so the cached build is reused). You'll get a new public link.
